# 38｜ChemVista 外部基线与 MS/MS 信息增益验证

本步骤只回答一个审稿人会直接追问的问题：**碎片级 MS/MS 相比只知道母离子质量和采集元数据，是否提供额外的外部预测信息？**

比较阶梯：`Development null → precursor m/z only → metadata only → Binned MS/MS → frozen final model`。其中 Binned MS/MS 严格沿用 Notebook 5 的 `fragment intensity + neutral loss + metadata` 路线，因此它与 metadata-only 的配对差值就是碎片/中性丢失信息的增量检验。主分析人群为 ChemVista `novel_all`（selected-novel + natural-novel），同时保留两个 novel 子集和 transfer 结果。

严格边界：本 notebook 不重新选模、不重新调参、不使用 ChemVista 标签校准模型。新增两个基线完全沿用 Notebook 5 的固定规格并仅在 Development 上训练；新增预测先写入无标签文件并计算哈希，之后才打开 ChemVista 终点标签。Notebook 20 的 Binned MS/MS 和最终模型预测只读复用。

> 运行前请切换到 **Python (env_ms)**。正式论文结果使用 1000 次配对 bootstrap。

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

assert Path(sys.prefix).name.lower() == 'env_ms', f'请切换到 Python (env_ms)；当前为 {sys.executable}'
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'outputs/pbt_msms_v1').exists()), None)
assert ROOT is not None, f'无法从 {cwd} 定位项目根目录'
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pbt_msms.chemvista_information_gain import analysis_paths, run_analysis

P = analysis_paths(ROOT)
print('Project root:', ROOT)
print('Python:', sys.executable)
print('Output:', P['out'])

## 1. 运行前只读核查

这里仅检查 Notebook 5 和 Notebook 20 的冻结状态及必要文件，不触碰任何标签或模型参数。

In [ ]:
simple_manifest = json.loads(P['simple_manifest'].read_text(encoding='utf-8'))
external_manifest = json.loads(P['ext_manifest'].read_text(encoding='utf-8'))
preflight = pd.DataFrame([
    {'item': 'Notebook 5 simple baselines', 'status': simple_manifest.get('qc_status'), 'expected': 'PASS'},
    {'item': 'Notebook 5 external locked', 'status': str(simple_manifest.get('external_test_unlocked')), 'expected': 'False'},
    {'item': 'Notebook 20 external evaluation', 'status': external_manifest.get('status'), 'expected': 'READY_FOR_NOTEBOOK_21'},
    {'item': 'Notebook 20 QC', 'status': str(external_manifest.get('qc_all_pass')), 'expected': 'True'},
])
display(preflight)
assert simple_manifest.get('qc_status') == 'PASS'
assert simple_manifest.get('external_test_unlocked') is False
assert external_manifest.get('status') == 'READY_FOR_NOTEBOOK_21'
assert external_manifest.get('qc_all_pass') is True

## 2. 正式运行

该单元会在 `outputs/pbt_msms_v1/chemvista_information_gain_38_v1` 写入独立结果，不修改第 5、19、20 步的文件。CPU 即可运行；主要耗时来自固定基线拟合与配对 bootstrap。

In [ ]:
N_BOOTSTRAP = 1000  # 正式论文值；仅调试流程时可临时改为 100
result = run_analysis(ROOT, n_bootstrap=N_BOOTSTRAP)
print(json.dumps({
    'status': result['manifest']['status'],
    'output_dir': str(result['output_dir']),
    'figure': str(result['figure']),
    'summary': str(result['summary']),
}, ensure_ascii=False, indent=2))

## 3. QC 与主结果

所有 QC 必须为 `True`。主表先看 `novel_all`：P 端点为 PR-AUC（越高越好），B/T 回归端点为 MAE（越低越好）。

In [ ]:
display(result['qc'])
assert result['qc']['passed'].all(), 'Notebook 38 QC 未全部通过'

primary = result['metrics'].loc[
    result['metrics']['cohort'].eq('novel_all'),
    ['task', 'pipeline', 'n', 'coverage', 'primary_metric', 'primary_value', 'brier', 'r2', 'spearman']
].copy()
display(primary)

## 4. 核心证据：相对 metadata-only 的配对增益

`observed_improvement > 0` 表示 MS/MS 路线更好。95% CI 全部高于 0 才能写成“明确增加信息”；CI 跨 0 应写成“方向为正但证据不确定”；CI 全部低于 0 则不能宣称该端点获益。P 端点同时检查 Brier improvement，避免只看排序能力而忽略概率校准。

In [ ]:
gain_main = result['gain'].loc[
    result['gain']['cohort'].eq('novel_all')
    & result['gain']['reference'].eq('Metadata only')
    & result['gain']['candidate'].isin(['Binned MS/MS', 'Frozen final model']),
    ['task', 'candidate', 'reference', 'metric', 'n', 'observed_improvement', 'ci_low', 'ci_high',
     'bootstrap_probability_improvement', 'one_sided_p']
].copy()
display(gain_main)

In [ ]:
display(Image(filename=str(result['figure']), width=1200))

## 5. 可直接用于写作的结果摘要

下面的摘要由实际结果自动生成，避免手工抄错数值。最终论文措辞必须服从置信区间，而不是只看点估计。

In [ ]:
display(Markdown(result['summary'].read_text(encoding='utf-8')))

## 论文中允许和不允许的结论

允许：在独立 ChemVista 新化合物上，冻结的 MS/MS 模型能够预测哪些 P/B/T 端点；并定量报告它相较于 precursor-only、metadata-only 和 Development null 的增益及不确定性。

不允许：把外部数据上的事后比较描述成新的模型选择；把“谱图可预测性质”写成“无需鉴定即可确定污染物身份”；或在置信区间跨 0 时宣称普遍优于 metadata-only。

结构模型仍是内部性能上限参照，不纳入本步的外部识别无关比较；本步专门隔离碎片谱在母离子和采集条件之外的增量信息。